In [1]:
from astropy.io import fits
import numpy as np
from skimage import exposure
import os
import cv2
import pandas as pd
from ultralytics import YOLO

In [2]:
input_directory = r"C:\Users\Windows\Downloads\2024_11_26_13_02_24_exp_0_600000_COSMOS1929"

output_directory = r"C:\Users\Windows\Downloads\GSOD_notebook"

model_path = r"C:\Users\Windows\Downloads\GSOD_notebook\GSOD_Streak.pt"

In [3]:
def brighten_image(image):
    brightened_image = np.clip(image * 1.5, 0, 65535).astype(np.uint16)
    equalized_image = exposure.equalize_hist(brightened_image)
    gamma_corrected_image = exposure.adjust_gamma(equalized_image, gamma=2.0)
    return gamma_corrected_image

In [4]:
def process_fits_images(input_dir):
    processed_images = {}

    for filename in os.listdir(input_dir):
        if filename.endswith(".fits"):
            fits_path = os.path.join(input_dir, filename)

            with fits.open(fits_path) as hdulist:
                img = hdulist[0].data

            brightened_img = brighten_image(img)

            image_8bit = (brightened_img / np.max(brightened_img) * 255).astype(np.uint8)
            image_rgb = cv2.cvtColor(image_8bit, cv2.COLOR_GRAY2RGB)

            processed_images[filename] = image_rgb

    return processed_images

In [5]:
def detect_streak(processed_images, model_path, output_folder):

    os.makedirs(output_folder, exist_ok=True)

    model = YOLO(model_path)
    results_list = []

    for filename, image in processed_images.items():
        original_image = image.copy()
        results = model.predict(source=image, save=False)

        object_count = 0

        for result in results:
            for box in result.boxes:
                object_count += 1

                x_center, y_center, width, height = box.xywh[0].tolist()

                x1 = int(x_center - width / 2)
                y1 = int(y_center - height / 2)
                x2 = int(x_center + width / 2)
                y2 = int(y_center + height / 2)

                center_x = int((x1 + x2) / 2)
                center_y = int((y1 + y2) / 2)

                confidence = float(box.conf[0])

                cv2.rectangle(original_image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.circle(original_image, (center_x, center_y), 5, (255, 0, 0), -1)

                cv2.putText(
                    original_image,
                    f"{confidence:.2f}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 255, 0),
                    1
                )

                results_list.append([
                    filename,
                    object_count,
                    x1, y1, x2, y2,
                    center_x, center_y,
                    confidence
                ])

        if object_count == 0:
            results_list.append([
                filename,
                0,
                None, None, None, None,
                None, None,
                None
            ])

        output_image_path = os.path.join(
            output_folder,
            filename.replace(".fits", ".png")
        )

        cv2.imwrite(output_image_path, original_image)

    df = pd.DataFrame(
        results_list,
        columns=[
            "File name",
            "Object ID",
            "x1", "y1", "x2", "y2",
            "Center_x", "Center_y",
            "Confidence"
        ]
    )

    return df

In [6]:
processed_images = process_fits_images(input_directory)

len(processed_images)

2

In [7]:
df_results = detect_streak(
    processed_images,
    model_path,
    output_directory
)

df_results.head()


0: 2048x2048 1 streak, 3415.4ms
Speed: 141.9ms preprocess, 3415.4ms inference, 8.2ms postprocess per image at shape (1, 3, 2048, 2048)

0: 2048x2048 1 streak, 2677.0ms
Speed: 89.5ms preprocess, 2677.0ms inference, 6.2ms postprocess per image at shape (1, 3, 2048, 2048)


,File name,Object ID,x1,y1,x2,y2,Center_x,Center_y,Confidence
0,2024_11_25_16_12_46_exp_10_000000_KAZSAT2.fits,1,2108,2029,2254,2195,2181,2112,0.925983
1,2024_11_26_13_02_24_exp_0_600000_COSMOS1929.fits,1,2511,2015,2916,2373,2713,2194,0.969404


In [8]:
csv_output_path = os.path.join(output_directory, "results.csv")

df_results.to_csv(csv_output_path, index=False)

print(f"Results saved to: {csv_output_path}")
print("Successfully completed.")

Results saved to: C:\Users\Windows\Downloads\GSOD_notebook\results.csv
Successfully completed.


# Export โมเดลเป็น ONNX

In [9]:
from ultralytics import YOLO

# Load the model
model = YOLO(model_path)

# Export the model to ONNX format
model.export(format="onnx")  # creates 'GSOD_Streak.onnx'

Ultralytics 8.4.17  Python-3.10.11 torch-2.7.0+cu118 CPU (Intel Core i3-7020U 2.30GHz)
YOLOv9t summary (fused): 197 layers, 1,970,979 parameters, 0 gradients, 7.6 GFLOPs

PyTorch: starting from 'C:\Users\Windows\Downloads\GSOD_notebook\GSOD_Streak.pt' with input shape (1, 3, 2048, 2048) BCHW and output shape(s) (1, 5, 86016) (4.9 MB)

ONNX: starting export with onnx 1.20.1 opset 19...
ONNX: slimming with onnxslim 0.1.86...
ONNX: export success  10.5s, saved as 'C:\Users\Windows\Downloads\GSOD_notebook\GSOD_Streak.onnx' (9.4 MB)

Export complete (17.0s)
Results saved to C:\Users\Windows\Downloads\GSOD_notebook
Predict:         yolo predict task=detect model=C:\Users\Windows\Downloads\GSOD_notebook\GSOD_Streak.onnx imgsz=2048 
Validate:        yolo val task=detect model=C:\Users\Windows\Downloads\GSOD_notebook\GSOD_Streak.onnx imgsz=2048 data=/home/mr-valentine/mysegment/datasets/2526_object_folder_PNG/dot_split/dataset.yaml  
Visualize:       https://netron.app


'C:\\Users\\Windows\\Downloads\\GSOD_notebook\\GSOD_Streak.onnx'

In [10]:
model_path_onnx = r"C:\Users\Windows\Downloads\GSOD_notebook\GSOD_Streak.onnx"

# ตัวอย่างการ นำ weight best.pt ของ GSOD ที่แปลงเป็น ONNX มาใช้งาน

In [11]:
df_results = detect_streak(
    processed_images,
    model_path_onnx,
    output_directory
)

df_results.head()

WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Loading C:\Users\Windows\Downloads\GSOD_notebook\GSOD_Streak.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.21.1 with CPUExecutionProvider

0: 2048x2048 1 streak, 2696.8ms
Speed: 79.7ms preprocess, 2696.8ms inference, 7.4ms postprocess per image at shape (1, 3, 2048, 2048)

0: 2048x2048 1 streak, 2877.3ms
Speed: 88.2ms preprocess, 2877.3ms inference, 5.3ms postprocess per image at shape (1, 3, 2048, 2048)


,File name,Object ID,x1,y1,x2,y2,Center_x,Center_y,Confidence
0,2024_11_25_16_12_46_exp_10_000000_KAZSAT2.fits,1,2108,2029,2254,2195,2181,2112,0.925983
1,2024_11_26_13_02_24_exp_0_600000_COSMOS1929.fits,1,2511,2015,2916,2373,2713,2194,0.969404


# ตัวอย่างการ import จาก YOLO26 ให้เป็น ONNX และแสดงผลลัพธ์

In [12]:
from ultralytics import YOLO

# Load the YOLO26 model
model = YOLO("yolo26n.pt")

# Export the model to ONNX format
model.export(format="onnx")  # creates 'yolo26n.onnx'

# Load the exported ONNX model
onnx_model = YOLO("yolo26n.onnx")

# Run inference
results = onnx_model("https://ultralytics.com/images/bus.jpg")

Ultralytics 8.4.17  Python-3.10.11 torch-2.7.0+cu118 CPU (Intel Core i3-7020U 2.30GHz)
YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.4 GFLOPs

PyTorch: starting from 'yolo26n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.3 MB)

ONNX: starting export with onnx 1.20.1 opset 19...


c:\Users\Windows\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\onnx\symbolic_opset9.py:5350: UserWarning: Exporting aten::index operator of advanced indexing in opset 19 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.86...
ONNX: export success  4.8s, saved as 'yolo26n.onnx' (9.5 MB)

Export complete (6.1s)
Results saved to C:\Users\Windows\Downloads\GSOD_notebook
Predict:         yolo predict task=detect model=yolo26n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolo26n.onnx imgsz=640 data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Loading yolo26n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.21.1 with CPUExecutionProvider

Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 c:\Users\Windows\Downloads\GSOD_notebook\bus.jpg: 640x640 4 persons, 1 bus, 197.1ms
Speed: 80.0ms preprocess, 197.1ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
